In [ ]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score


df = pd.read_csv("data/1/train.csv")

X = df.drop(columns=["Label"])
y = df["Label"]


pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__kernel": ["linear", "rbf"],
    "svm__gamma": ["scale", 0.01, 0.001]
}


outer_cv = KFold(n_splits=10, shuffle=True, random_state=42)

results = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), start=1):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    inner_cv = KFold(n_splits=9, shuffle=True, random_state=42)

    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=inner_cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    preds = best_model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")

    results.append({
        "Fold": fold,
        "Accuracy": acc,
        "F1": f1,
        "Parameters": grid.best_params_
    })


results_df = pd.DataFrame(results)

print(results_df)

print("\nAccuracy mean ± std:",
      f"{results_df['Accuracy'].mean():.4f} ± {results_df['Accuracy'].std():.4f}")

print("F1 mean ± std:",
      f"{results_df['F1'].mean():.4f} ± {results_df['F1'].std():.4f}")


results_df.to_csv("svm_results.csv", index=False)
